# ExplainEthics Analysis
Analyse results produced by `cot_met_explain_ethics.py`.

**Pipeline:**
```
cot_met_explain_ethics.py  →  all_outs_cot_met_explain_ethics_*.pkl
                                        ↓
                        this notebook (load & evaluate)
```

**`all_outs` dict structure** (keyed by `'explainethicsN.cnf'`):
```
all_outs[name] = (vv, solout, bbout, missed_flag, rule_scores, cot_flag, scs, prompts)
  [0] vv          – list of inferred variable tuples
  [1] solout      – {'pos': [...], 'neg': [...]}  final backbone
  [2] bbout       – backbone after last iteration
  [3] missed_flag – True if the example was missed/skipped
  [4] rule_scores – dict of rule → score
  [5] cot_flag    – True if resolved by CoT instead of backbone
  [6] scs         – list of running probability tensors
  [7] prompts     – list of prompts used
```

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import pickle as pkl
import json
import csv
import os
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['font.size'] = 12

# ── Paths (adjust if needed) ───────────────────────────────────────────────────
BASE_PATH   = '/mnt/c/Tugas_Akhir/ARGOS_public_anon'
DATASET_PATH = BASE_PATH + '/SAT-LM/data/explainethics_test.json'
DIMACS_DIR   = BASE_PATH + '/main/dimacs'
CSV_PATH     = BASE_PATH + '/main/dimacs_csvs/solver_finished.csv'
LABELS_CSV   = BASE_PATH + '/main/explain_ethics_labels.csv'

# Path to the pkl saved by cot_met_explain_ethics.py (fill in manually or glob)
# Example: all_outs_cot_met_explain_ethics_explain_ethics_rulethresh=0.3_....pkl
import glob
pkl_files = glob.glob(BASE_PATH + '/all_outs_cot_met_explain_ethics_*.pkl')
print('Available pkl files:')
for f in pkl_files:
    print(' ', f)

In [ ]:
# ── Load pkl (pick the one you want) ──────────────────────────────────────────
PKL_PATH = pkl_files[0]  # change index or hardcode path
outs = pkl.load(open(PKL_PATH, 'rb'))
print(f'Loaded {len(outs)} examples from {os.path.basename(PKL_PATH)}')

In [ ]:
# ── Load dataset ───────────────────────────────────────────────────────────────
with open(DATASET_PATH, 'r') as f:
    data = json.load(f)
print(f'Dataset: {len(data)} examples')

# Build index → gold_foundation mapping (true moral norm, not the decoy label)
gold = {i: d['gold_foundation'] for i, d in enumerate(data)}
gt   = {i: d['gt']              for i, d in enumerate(data)}

In [ ]:
# ── Load labels dict (from CSV written by explain_ethics_to_sat.py) ────────────
labels = {}
if os.path.exists(LABELS_CSV):
    with open(LABELS_CSV, 'r') as lf:
        lr = csv.reader(lf)
        for row in lr:
            # row[0] = 'explainethicsN.py', row[1] = norm label
            cnf_name = row[0].replace('.py', '.cnf')
            labels[cnf_name] = row[1].strip().lower()
    print(f'Loaded {len(labels)} labels from explain_ethics_labels.csv')
else:
    # Fallback: derive from dataset index in filename
    for name in outs.keys():
        try:
            idx = int(name.replace('explainethics', '').split('.')[0])
            labels[name] = data[idx]['gold_foundation']
        except Exception:
            pass
    print(f'Derived {len(labels)} labels from dataset')

# Normalise label whitespace
labels = {k: v.strip() for k, v in labels.items()}
print('Sample:', dict(list(labels.items())[:3]))

In [ ]:
# ── Build preds dict from all_outs ────────────────────────────────────────────
#
# solout = all_outs[name][1]  →  {'pos': [...lits...], 'neg': [...lits...]}
#   pos=[], neg has literals  → backbone says pos is UNSAT → norm NOT violated → pred='false'
#   neg=[], pos has literals  → backbone says neg is UNSAT → norm IS violated  → pred='true'
#   cot_flag=True             → resolved by CoT; scs[-1].argmax()==0 → Yes
#
preds = {}
cot_count  = 0
sat_count  = 0
miss_count = 0

for name, value in outs.items():
    vv, solout, bbout, missed_flag, rule_scores, cot_flag, scs, prompts = value

    if missed_flag:
        preds[name] = 'missed'
        miss_count += 1
        continue

    if cot_flag:
        # CoT path: last sc tensor votes[0]=yes, votes[1]=no
        cot_count += 1
        if scs:
            preds[name] = 'true' if scs[-1].argmax() == 0 else 'false'
        else:
            preds[name] = 'missed'
    else:
        # Backbone SAT path
        sat_count += 1
        if len(solout['neg']) == 0:   # neg UNSAT → norm violated
            preds[name] = 'true'
        elif len(solout['pos']) == 0: # pos UNSAT → norm NOT violated
            preds[name] = 'false'
        else:
            preds[name] = 'missed'    # underdetermined even after loop

print(f'Total examples: {len(preds)}')
print(f'  SAT backbone: {sat_count}')
print(f'  CoT fallback: {cot_count}')
print(f'  Missed:       {miss_count}')

In [ ]:
# ── Accuracy ──────────────────────────────────────────────────────────────────
acc     = 0
missed  = 0
correct_by_cot = 0
correct_by_sat = 0

for name, pred in preds.items():
    label = labels.get(name, '').strip().lower()
    if pred == 'missed':
        missed += 1
        continue

    # For ExplainEthics: compare predicted norm-violation boolean against gt
    try:
        idx = int(name.replace('explainethics', '').split('.')[0])
        true_gt = data[idx]['gt']  # 'true' or 'false'
    except Exception:
        true_gt = label

    correct = (pred == true_gt.lower().strip())
    if correct:
        acc += 1
        _, _, _, _, _, cot_flag, _, _ = outs[name]
        if cot_flag:
            correct_by_cot += 1
        else:
            correct_by_sat += 1

total = max(len(preds) - missed, 1)
print(f'Overall accuracy : {acc/total:.4f}  ({acc}/{total})')
print(f'Correct via SAT  : {correct_by_sat}')
print(f'Correct via CoT  : {correct_by_cot}')
print(f'Missed/skipped   : {missed}')

In [ ]:
# ── Per-prediction breakdown ───────────────────────────────────────────────────
print(f'{'Name':<30} {'Pred':<8} {'GT':<8} {'OK?'}')
print('-'*55)
for name, pred in preds.items():
    try:
        idx   = int(name.replace('explainethics', '').split('.')[0])
        true_gt = data[idx]['gt']
    except Exception:
        true_gt = '??'
    ok = '✓' if pred == true_gt.lower().strip() else '✗'
    print(f'{name:<30} {pred:<8} {true_gt:<8} {ok}')

In [ ]:
# ── Confusion matrix (predicted norm category vs gold norm) ───────────────────
# Only meaningful once multiple examples are run
from collections import defaultdict

norm_labels = ['violate_care', 'violate_fairness', 'violate_loyalty',
               'violate_authority', 'violate_sanctity', 'violate_liberty']

# For each example, get which norm was the shown label (decoy or correct)
# and what the gold norm actually was
confusion = defaultdict(lambda: defaultdict(int))  # confusion[shown][gold]
for name, pred in preds.items():
    try:
        idx = int(name.replace('explainethics', '').split('.')[0])
        shown_label     = data[idx]['label']          # may be decoy
        gold_foundation = data[idx]['gold_foundation'] # always correct
        true_gt = data[idx]['gt']                     # 'true'/'false'
    except Exception:
        continue

    if pred == 'missed':
        continue

    # A correct pred=true means model agreed with shown label
    pred_norm = shown_label if pred == 'true' else 'rejected_' + shown_label
    confusion[pred_norm][gold_foundation] += 1

print('Confusion (pred_decision → gold_foundation):')
for k, v in confusion.items():
    print(f'  {k}: {dict(v)}')

In [ ]:
# ── CoT vs SAT accuracy comparison ────────────────────────────────────────────
cot_correct, cot_total = 0, 0
sat_correct, sat_total = 0, 0

for name, pred in preds.items():
    if pred == 'missed': continue
    try:
        idx = int(name.replace('explainethics', '').split('.')[0])
        true_gt = data[idx]['gt'].lower()
    except Exception:
        continue

    _, _, _, _, _, cot_flag, _, _ = outs[name]
    correct = (pred == true_gt)
    if cot_flag:
        cot_total += 1
        cot_correct += correct
    else:
        sat_total += 1
        sat_correct += correct

print(f'SAT backbone accuracy: {sat_correct}/{sat_total} = {sat_correct/max(sat_total,1):.3f}')
print(f'CoT fallback accuracy: {cot_correct}/{cot_total} = {cot_correct/max(cot_total,1):.3f}')

In [ ]:
# ── Bar chart: correct vs incorrect ───────────────────────────────────────────
categories = ['Correct (SAT)', 'Correct (CoT)', 'Incorrect', 'Missed']
incorrect = total - acc
values = [correct_by_sat, correct_by_cot, incorrect, missed]
colors = ['#2ecc71', '#27ae60', '#e74c3c', '#95a5a6']

plt.figure(figsize=(8, 4))
bars = plt.bar(categories, values, color=colors, edgecolor='white', linewidth=1.5)
for bar, val in zip(bars, values):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
             str(val), ha='center', va='bottom', fontweight='bold')
plt.title('ExplainEthics Pipeline Results')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('explainethics_results.png', dpi=150)
plt.show()

In [ ]:
# ── Inspect a single example in detail ────────────────────────────────────────
name = list(outs.keys())[0]   # change to inspect a specific name
val  = outs[name]
vv, solout, bbout, missed_flag, rule_scores, cot_flag, scs, prompts = val

try:
    idx  = int(name.replace('explainethics', '').split('.')[0])
    d    = data[idx]
    print(f'Context   : {d["context"]}')
    print(f'Shown norm: {d["label"]}')
    print(f'Gold norm : {d["gold_foundation"]}')
    print(f'GT (true violation?): {d["gt"]}')
except Exception as e:
    print(f'Could not look up example: {e}')

print(f'\nPrediction : {preds.get(name, "N/A")}')
print(f'Resolved by: {"CoT" if cot_flag else "SAT backbone"}')
print(f'solout     : {solout}')
print(f'Rule scores: {rule_scores}')
print(f'Inferred vars (vv): {vv}')